# Decision Tree - Model Training

Entrenamiento de modelos de Árbol de Decisión (BALANCED vs RAW) con GridSearchCV.

> **¿Por qué Decision Tree?**  
> Recomendado por el tutor académico dado el tamaño limitado del dataset (699 pacientes). Los árboles de decisión son altamente interpretables, no requieren escala de variables, y con la profundidad correctamente limitada son robustos frente al sobreajuste en datasets pequeños. Su estructura de reglas if-then facilita la validación clínica directa por parte del equipo médico.

## 1. Imports

In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_fscore_support
)
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


## 2. Configure Paths

In [2]:
BASE_PATH = Path('../../..').resolve()
DATA_PATH = BASE_PATH / 'data' / 'processed' / 'pickle'
MODELS_PATH = BASE_PATH / 'models' / 'decision_tree'
RESULTS_PATH = BASE_PATH / 'data' / 'results' / 'decision_tree'

MODELS_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"✓ Paths configured")
print(f"  Models  → {MODELS_PATH}")
print(f"  Results → {RESULTS_PATH}")

✓ Paths configured
  Models  → /home/pablo/Desktop/Estressss/pdg/models/decision_tree
  Results → /home/pablo/Desktop/Estressss/pdg/data/results/decision_tree


## 3. Load Data

In [3]:
with open(DATA_PATH / 'X_train_balanced.pkl', 'rb') as f:
    X_train_balanced = pickle.load(f)
with open(DATA_PATH / 'y_train_balanced.pkl', 'rb') as f:
    y_train_balanced = pickle.load(f)

with open(DATA_PATH / 'X_train_raw.pkl', 'rb') as f:
    X_train_raw = pickle.load(f)
with open(DATA_PATH / 'y_train_raw.pkl', 'rb') as f:
    y_train_raw = pickle.load(f)

if isinstance(X_train_balanced, pd.DataFrame): X_train_balanced = X_train_balanced.values
if isinstance(X_train_raw,   pd.DataFrame): X_train_raw   = X_train_raw.values
if isinstance(y_train_balanced, pd.DataFrame): y_train_balanced = y_train_balanced.iloc[:, 0]
if isinstance(y_train_raw,   pd.DataFrame): y_train_raw   = y_train_raw.iloc[:, 0]

print(f"✓ BALANCED Training set : {X_train_balanced.shape}")
print(f"✓ RAW   Training set : {X_train_raw.shape}")
print(f"Class distribution:")
print(f"  BALANCED Train : {dict(pd.Series(y_train_balanced).value_counts().sort_index())}")
print(f"  RAW   Train : {dict(pd.Series(y_train_raw).value_counts().sort_index())}")

✓ BALANCED Training set : (898, 651)
✓ RAW   Training set : (699, 651)
Class distribution:
  BALANCED Train : {0: np.int64(633), 1: np.int64(265)}
  RAW   Train : {0: np.int64(633), 1: np.int64(66)}


## 4. Hyperparameter Tuning - Grid Search

> **Parámetros clave de Decision Tree:**
>
> - **max_depth**: Profundidad máxima del árbol. Valor bajo = modelo más simple, menos sobreajuste. Crítico con datasets pequeños.
> - **min_samples_split**: Mínimo de muestras para dividir un nodo. Mayor valor = árbol más conservador.
> - **min_samples_leaf**: Mínimo de muestras en una hoja. Evita hojas con muy pocos pacientes, mejora generalización.
> - **criterion**: Función de impureza (gini o entropy). Ambas se evalúan para encontrar la mejor partición.
> - **class_weight**:  compensa el desbalance asignando mayor peso a las clases minoritarias.
>
> ⚡ **Grid Search moderado** — Decision Trees convergen rápido, el tiempo total será menor que RF o XGBoost.

In [4]:
print("=" * 70)
print("HYPERPARAMETER TUNING - GRID SEARCH")
print("=" * 70)
print("Configuration:")
print("  • Metric: Recall Macro (10-fold Stratified CV)")
print("  • Cross-validation: StratifiedKFold (n_splits=10)")
print("  • Strategy: Tune max_depth, min_samples_split, min_samples_leaf, criterion")

cv_stratified = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Parámetros específicos de Decision Tree
# max_depth limitado intencionalmente para evitar sobreajuste con 699 pacientes
param_grid = {
    'max_depth':         [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf':  [1, 2, 4, 8],
    'criterion':         ['gini', 'entropy'],
}

print(f"Parameter grid:")
for k, v in param_grid.items():
    print(f"  {k}: {v}")
print(f"  Total combinations: {5*4*4*2} × 10 folds = {5*4*4*2*10} fits")

HYPERPARAMETER TUNING - GRID SEARCH
Configuration:
  • Metric: Recall Macro (10-fold Stratified CV)
  • Cross-validation: StratifiedKFold (n_splits=10)
  • Strategy: Tune max_depth, min_samples_split, min_samples_leaf, criterion
Parameter grid:
  max_depth: [3, 5, 7, 10, None]
  min_samples_split: [2, 5, 10, 20]
  min_samples_leaf: [1, 2, 4, 8]
  criterion: ['gini', 'entropy']
  Total combinations: 160 × 10 folds = 1600 fits


## 5. Base Model and Grid Search Function

In [5]:
def create_base_model():
    """Base Decision Tree model.
    
    - class_weight='balanced': compensa el desbalance de clases asignando
      pesos inversamente proporcionales a la frecuencia de cada clase.
      Esencial dado que clase 0 representa ~90% de los datos.
    - random_state=42: reproducibilidad de resultados.
    """
    return DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42,
    )


def run_grid_search(X_train, y_train, model_name):
    """Execute GridSearchCV for BALANCED or RAW model."""
    print(f"{chr(45)*70}")
    print(f"Grid Search: {model_name} Model")
    print(f"{chr(45)*70}")

    # compute_sample_weight refuerza el peso de clases minoritarias
    sample_weights = compute_sample_weight('balanced', y_train)

    grid_search = GridSearchCV(
        estimator=create_base_model(),
        param_grid=param_grid,
        cv=cv_stratified,
        scoring='recall_macro',   # métrica clínica prioritaria
        n_jobs=-1,
        verbose=1,
    )

    print(f"Searching optimal hyperparameters...")
    grid_search.fit(X_train, y_train, sample_weight=sample_weights)

    print(f"✓ Best hyperparameters ({model_name}):")
    for param, value in grid_search.best_params_.items():
        print(f"    {param}: {value}")
    print(f"✓ Best Recall Macro (CV): {grid_search.best_score_:.4f}")

    cv_results = pd.DataFrame(grid_search.cv_results_)
    suffix = 'balanced' if 'BALANCED' in model_name else 'raw'
    cv_results.to_csv(RESULTS_PATH / f'gridsearch_results_{suffix}.csv', index=False)
    print(f"✓ Grid Search results saved")

    return grid_search.best_estimator_, grid_search

## 6. Run Grid Search for Both Models

In [6]:
model_balanced, gs_balanced = run_grid_search(X_train_balanced, y_train_balanced, 'BALANCED')
model_raw,   gs_raw   = run_grid_search(X_train_raw,   y_train_raw,   'RAW')

print(f"" + "="*70)
print(f"✓ Both models trained successfully")
print(f"="*70)

----------------------------------------------------------------------
Grid Search: BALANCED Model
----------------------------------------------------------------------
Searching optimal hyperparameters...
Fitting 10 folds for each of 160 candidates, totalling 1600 fits


✓ Best hyperparameters (BALANCED):
    criterion: entropy
    max_depth: None
    min_samples_leaf: 1
    min_samples_split: 2
✓ Best Recall Macro (CV): 0.8680
✓ Grid Search results saved
----------------------------------------------------------------------
Grid Search: RAW Model
----------------------------------------------------------------------
Searching optimal hyperparameters...
Fitting 10 folds for each of 160 candidates, totalling 1600 fits
✓ Best hyperparameters (RAW):
    criterion: entropy
    max_depth: 5
    min_samples_leaf: 8
    min_samples_split: 2
✓ Best Recall Macro (CV): 0.7371
✓ Grid Search results saved
✓ Both models trained successfully


## 7. Training Set Performance

> ⚠️ Decision Trees tienden a sobreajustarse y memorizar el training set completamente si max_depth no está limitado. Si ves accuracy=1.0, es normal y esperado — lo importante son las métricas en Cross-Validation (notebook de evaluación).

In [7]:
print("" + "="*70)
print("TRAINING SET PERFORMANCE")
print("="*70)

y_train_pred_balanced = model_balanced.predict(X_train_balanced)
print(f"BALANCED Model:")
print(f"  Accuracy : {accuracy_score(y_train_balanced, y_train_pred_balanced):.4f}")
print(f"  Precision: {precision_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")
print(f"  F1-Score : {f1_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")

y_train_pred_raw = model_raw.predict(X_train_raw)
print(f"RAW Model:")
print(f"  Accuracy : {accuracy_score(y_train_raw, y_train_pred_raw):.4f}")
print(f"  Precision: {precision_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")
print(f"  F1-Score : {f1_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")

TRAINING SET PERFORMANCE
BALANCED Model:
  Accuracy : 1.0000
  Precision: 1.0000
  Recall   : 1.0000
  F1-Score : 1.0000
RAW Model:
  Accuracy : 0.6309
  Precision: 0.6019
  Recall   : 0.7962
  F1-Score : 0.5413


## 8. Per-Class Metrics (Training Set)

In [8]:
print("" + "="*70)
print("PER-CLASS METRICS (TRAINING SET)")
print("="*70)

prec_s, rec_s, f1_s, sup_s = precision_recall_fscore_support(y_train_balanced, y_train_pred_balanced, average=None)
print(f"BALANCED Model:")
print(pd.DataFrame({'Class': [0,1], 'Precision': prec_s, 'Recall': rec_s, 'F1-Score': f1_s, 'Support': sup_s}).to_string(index=False))

prec_r, rec_r, f1_r, sup_r = precision_recall_fscore_support(y_train_raw, y_train_pred_raw, average=None)
print(f"RAW Model:")
print(pd.DataFrame({'Class': [0,1], 'Precision': prec_r, 'Recall': rec_r, 'F1-Score': f1_r, 'Support': sup_r}).to_string(index=False))

PER-CLASS METRICS (TRAINING SET)
BALANCED Model:
 Class  Precision  Recall  F1-Score  Support
     0        1.0     1.0       1.0      633
     1        1.0     1.0       1.0      265
RAW Model:
 Class  Precision   Recall  F1-Score  Support
     0   1.000000 0.592417  0.744048      633
     1   0.203704 1.000000  0.338462       66


## 9. Save Models

In [9]:
with open(MODELS_PATH / 'modelo_decision_tree_balanced.pkl', 'wb') as f:
    pickle.dump(model_balanced, f)
print(f"✓ BALANCED model saved")

with open(MODELS_PATH / 'modelo_decision_tree_raw.pkl', 'wb') as f:
    pickle.dump(model_raw, f)
print(f"✓ RAW model saved")

print(f"✅ Both models ready for evaluation")
print(f"   → Run decision_tree_evaluation.ipynb next")

✓ BALANCED model saved
✓ RAW model saved
✅ Both models ready for evaluation
   → Run decision_tree_evaluation.ipynb next
